# 🎬 Supernan Hindi Dubbing Pipeline

**Zero-cost pipeline: Hindi dubbed video with lip sync**

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

Pipeline stages:
1. Install dependencies
2. Clone VideoReTalking & GFPGAN
3. Upload & extract 15-second clip
4. Whisper transcription
5. IndicTrans2 Hindi translation
6. Coqui XTTS v2 voice cloning
7. Audio speed adjustment
8. VideoReTalking lip-sync
9. GFPGAN face enhancement
10. Download output

**1: Check GPU**

In [ ]:
import subprocess, torch
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else 'No NVIDIA GPU.')
except FileNotFoundError:
    print('No nvidia-smi (Mac/CPU mode).')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


**2: Install System Deps**

In [ ]:
!apt-get install -y ffmpeg wget unzip -q
print('✓ ffmpeg installed')

**3: Install Python packages**

In [ ]:
# PyTorch with CUDA (Colab already has this, but ensure correct version)
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118

# Core pipeline deps
!pip install -q openai-whisper
!pip install -q TTS
!pip install -q transformers sentencepiece sacremoses
!pip install -q git+https://github.com/VarunGumma/IndicTransTokenizer.git@main@main
!pip install -q pydub librosa soundfile deep-translator
!pip install -q basicsr facexlib gfpgan realesrgan
print('✓ All packages installed')

**4: Clone Repos**

In [ ]:
import os

# VideoReTalking
if not os.path.exists('VideoReTalking'):
    !git clone -q https://github.com/vinthony/video-retalking VideoReTalking
    !pip install -q -r VideoReTalking/requirements.txt
print('✓ VideoReTalking ready')

# GFPGAN
if not os.path.exists('GFPGAN'):
    !git clone -q https://github.com/TencentARC/GFPGAN GFPGAN
    !pip install -q basicsr facexlib gfpgan
    %cd GFPGAN
    !python setup.py develop -q
    %cd ..
print('✓ GFPGAN ready')

**5: Download VideoReTalking Weights**

In [ ]:
import os
os.makedirs('models/VideoReTalking', exist_ok=True)

CHECKPOINTS = {
    '30_net_gen.pth':   'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/30_net_gen.pth',
    'DNet.pt':          'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/DNet.pt',
    'ENet.pth':         'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/ENet.pth',
    'GFPGANv1.3.pth':   'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/GFPGANv1.3.pth',
    'GPEN-BFR-512.pth': 'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/GPEN-BFR-512.pth',
    'ParseNet-latest.pth': 'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/ParseNet-latest.pth',
    'RetinaFace-R50.pth':  'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/RetinaFace-R50.pth',
    'shape_predictor_68_face_landmarks.dat': 'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/shape_predictor_68_face_landmarks.dat',
    'BFM.zip': 'https://github.com/vinthony/video-retalking/releases/download/v0.0.1/BFM.zip',
}

def check_ready():
    for fname in CHECKPOINTS.keys():
        if fname != 'BFM.zip' and not os.path.exists(f'models/VideoReTalking/{fname}'): return False
    return os.path.isdir('models/VideoReTalking/BFM')

if check_ready():
    print('✓ All VideoReTalking weights are already present (Fast skip enabled)')
else:
    for fname, url in CHECKPOINTS.items():
        dest = f'models/VideoReTalking/{fname}'
        if not os.path.exists(dest):
            print(f'Downloading {fname}...')
            !wget -c -q --show-progress -O {dest} {url}
        else:
            print(f'✓ {fname} already downloaded')
    if not os.path.isdir('models/VideoReTalking/BFM'):
        !unzip -q -o models/VideoReTalking/BFM.zip -d models/VideoReTalking/

print('✓ All VideoReTalking weights ready')


**6: Download GFPGAN Weights**

In [ ]:
os.makedirs('GFPGAN/experiments/pretrained_models', exist_ok=True)
gfpgan_weight = 'GFPGAN/experiments/pretrained_models/GFPGANv1.4.pth'
if not os.path.exists(gfpgan_weight):
    !wget -q --show-progress -O {gfpgan_weight} https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth
print('✓ GFPGAN weights ready')

**7: Download Input Video from Google Drive**

In [ ]:
import os, subprocess

# ── Paste your Google Drive share link here ───────────────────────────────────
GDRIVE_URL = 'https://drive.google.com/file/d/1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW/view?usp=sharing'

INPUT_VIDEO = 'supernan_training.mp4'

if not os.path.exists(INPUT_VIDEO):
    print('Downloading video from Google Drive...')
    # Install gdown if needed
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    gdown.download(url=GDRIVE_URL, output=INPUT_VIDEO, fuzzy=True)

if os.path.exists(INPUT_VIDEO):
    size_mb = os.path.getsize(INPUT_VIDEO) / 1e6
    print(f'✓ Video ready: {INPUT_VIDEO} ({size_mb:.1f} MB)')
else:
    print('⚠️  Download failed. If the file is restricted, share it as "Anyone with the link"')
    print('   Or place the video manually in:', os.getcwd())


**8: Configuration**

In [ ]:
# ⬇️ Change these if you want a different segment
START_SEC = 30
END_SEC   = 50

os.makedirs('workspace', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('models', exist_ok=True)

print(f'Processing segment: {START_SEC}s – {END_SEC}s ({END_SEC - START_SEC}s clip)')

**9: Stage 1 — Extract Clip**

In [ ]:
CLIP = 'workspace/clip.mp4'
duration = END_SEC - START_SEC

!ffmpeg -y -ss {START_SEC} -i "{INPUT_VIDEO}" -t {duration} \
    -c:v libx264 -c:a aac -preset fast -crf 18 {CLIP} -loglevel warning

from IPython.display import Video
print('✓ Clip extracted')
Video(CLIP, width=640)

**10: Stage 2 — Extract & Clean Reference Audio**

In [ ]:
import os
if 'CLIP' not in globals(): CLIP = 'workspace/clip.mp4'

AUDIO_16K = 'workspace/clip_audio_16k.wav'
AUDIO_REF = 'workspace/clip_audio_ref22k.wav'

print('Extracting and cleaning audio...')
# 1. Extract 16k for Whisper
!ffmpeg -y -i {CLIP} -vn -acodec pcm_s16le -ar 16000 -ac 1 {AUDIO_16K} -loglevel warning

# 2. Extract 22k Reference for XTTS with Denoising
filters = "afftdn,highpass=f=80,loudnorm=I=-14:LRA=7:TP=-1"
!ffmpeg -y -i {CLIP} -vn -af "{filters}" -ac 1 -ar 22050 {AUDIO_REF} -loglevel warning

print(f'✓ Audio extracted & cleaned.')
print(f'  16kHz (Transcription): {AUDIO_16K}')
print(f'  22kHz (Voice Clone Ref): {AUDIO_REF}')


**11: Stage 3 — Transcribe with Whisper**

In [ ]:
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import whisper

print('Loading Whisper medium model (downloading ~1.5 GB on first run)...')
model = whisper.load_model('medium')  # 'small' is faster; 'large' is more accurate

result = model.transcribe(AUDIO_16K, language='en', word_timestamps=True, verbose=False)
ENGLISH_TEXT = result['text'].strip()

with open('workspace/transcript_en.txt', 'w') as f:
    f.write(ENGLISH_TEXT)

print(f'✓ Transcript:\n{ENGLISH_TEXT}')

**12: Stage 4 — Translate to Hindi (IndicTrans2)**

In [ ]:
import torch, os
if 'DEVICE' not in globals(): DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if 'ENGLISH_TEXT' not in globals():
    if os.path.exists('workspace/transcript_en.txt'):
        with open('workspace/transcript_en.txt', 'r') as f: ENGLISH_TEXT = f.read().strip()
        print('✓ ENGLISH_TEXT recovered from disk')
    else: raise NameError('ENGLISH_TEXT missing. Run Cell 11.')

def translate_indictrans2(text):
    try:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        from IndicTransToolkit import IndicProcessor
        MODEL = 'ai4bharat/indictrans2-en-indic-1B'
        print('Loading IndicTrans2...')
        tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, trust_remote_code=True).to(DEVICE)
        ip = IndicProcessor(inference=True)

        sentences = [s.strip() for s in text.split('.') if s.strip()]
        batch = ip.preprocess_batch(sentences, src_lang='eng_Latn', tgt_lang='hin_Deva')
        inputs = tokenizer(batch, truncation=True, padding='longest', return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, num_beams=5, max_length=256)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        postprocessed = ip.postprocess_batch(decoded, lang='hin_Deva')
        return ' '.join(postprocessed)
    except Exception as e:
        print(f'IndicTrans2 failed: {e}. Falling back to Google Translate...')
        return None

print('Attempting translation...')
HINDI_TEXT = translate_indictrans2(ENGLISH_TEXT)
if not HINDI_TEXT:
    from deep_translator import GoogleTranslator
    HINDI_TEXT = GoogleTranslator(source='en', target='hi').translate(ENGLISH_TEXT)

with open('workspace/translation_hi.txt', 'w', encoding='utf-8') as f: f.write(HINDI_TEXT)
print(f'✓ Hindi translation completed.')


**13: Stage 5 — Smart XTTS v2 Voice Cloning**

In [ ]:
import os, re, torch
from TTS.api import TTS

# Global State Recovery & Guards
if 'DEVICE' not in globals(): DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if 'AUDIO_REF' not in globals(): AUDIO_REF = 'workspace/clip_audio_ref22k.wav'
if 'HINDI_TEXT' not in globals() or not HINDI_TEXT.strip():
    if os.path.exists('workspace/translation_hi.txt'):
        with open('workspace/translation_hi.txt', 'r', encoding='utf-8') as f: HINDI_TEXT = f.read().strip()
    if not HINDI_TEXT.strip():
        if 'ENGLISH_TEXT' in globals() and ENGLISH_TEXT: HINDI_TEXT = ENGLISH_TEXT
        else: raise ValueError("HINDI_TEXT is empty. Run Cell 12.")

os.environ['COQUI_TOS_AGREED'] = '1'
TTS_RAW = 'workspace/tts_raw.wav'
TTS_CLEAN = 'workspace/tts_clean.wav'
os.makedirs('workspace/tts_chunks', exist_ok=True)

print('Loading Coqui XTTS v2...')
if 'tts' not in globals():
    tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2')
    if DEVICE == 'cuda': tts = tts.cuda()

def smart_split_hindi(text, max_len=100):
    """Splits text at conjunctions and punctuation for stability."""
    import re
    # Split at punctuation
    text = re.sub(r'([।,.!?])', r'\1|', text)
    # Split at common Hindi conjunctions
    for conj in [' क्योंकि ', ' इसलिए ', ' या ', ' और ', ' लेकिन ', ' तो ', ' कि ']:
        text = text.replace(conj, f'|{conj}')
    
    raw_parts = [p.strip() for p in text.split('|') if p.strip()]
    chunks, curr = [], ""
    for p in raw_parts:
        if len(curr) + len(p) < max_len:
            curr += " " + p if curr else p
        else:
            if curr: chunks.append(curr.strip())
            curr = p
    if curr: chunks.append(curr.strip())
    return chunks

chunks = smart_split_hindi(HINDI_TEXT)
print(f'Synthesizing {len(chunks)} chunk(s)...')

chunk_paths = []
for i, chunk in enumerate(chunks):
    p = f'workspace/tts_chunks/tts_{i:04d}.wav'
    print(f'  [{i+1}/{len(chunks)}] {chunk[:50]}...')
    tts.tts_to_file(text=chunk, speaker_wav=AUDIO_REF, language='hi', file_path=p, split_sentences=False)
    chunk_paths.append(p)

# Concat with natural breaths (100ms silence)
if len(chunk_paths) > 1:
    !ffmpeg -y -f lavfi -i anullsrc=r=22050:cl=mono -t 0.1 workspace/sil.wav -loglevel warning
    with open('workspace/concat_list.txt', 'w') as f:
        for cp in chunk_paths:
            f.write(f"file '{os.path.abspath(cp)}'\n")
            f.write(f"file '{os.path.abspath('workspace/sil.wav')}'\n")
    !ffmpeg -y -f concat -safe 0 -i workspace/concat_list.txt -c copy {TTS_RAW} -loglevel warning
else:
    import shutil
    shutil.copy(chunk_paths[0], TTS_RAW)

print('Applying Ultimate Clarity Booster...')
filters = (
    "highpass=f=200,"
    "equalizer=f=3000:t=q:w=1:g=5,"
    "equalizer=f=6000:t=q:w=1:g=5,"
    "treble=g=5:f=8000,"
    "acompressor=threshold=0.08:ratio=4:attack=5:release=50:makeup=2,"
    "loudnorm=I=-14:LRA=7:TP=-1"
)
!ffmpeg -y -i {TTS_RAW} -af "{filters}" -ar 44100 {TTS_CLEAN} -loglevel warning

print(f'✓ Filtered & Balanced Dub: {TTS_CLEAN}')
TTS_FINAL_AUDIO = TTS_CLEAN


**14: Stage 6 — Natural Speed-Locked Sync**

In [ ]:
import subprocess, os

# State Recovery
if 'CLIP' not in globals(): CLIP = 'workspace/clip.mp4'
if 'TTS_FINAL_AUDIO' in globals(): TTS_RAW = TTS_FINAL_AUDIO
elif os.path.exists('workspace/tts_clean.wav'): TTS_RAW = 'workspace/tts_clean.wav'
else: TTS_RAW = 'workspace/tts_raw.wav'

TTS_ADJ = 'workspace/tts_adjusted.wav'

def get_duration(path):
    if not os.path.exists(path): return 0.0
    r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',path],
                       capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return 0.0

tts_dur = get_duration(TTS_RAW)
clip_dur = get_duration(CLIP)

print(f'Syncing {TTS_RAW} ({tts_dur:.2f}s) to {CLIP} ({clip_dur:.2f}s)')

if tts_dur > 0 and clip_dur > 0:
    ratio = tts_dur / clip_dur
    # Limit speedup to 1.15x to avoid 'chipmunk' effect
    safe_ratio = min(ratio, 1.15)
    
    if ratio > 1.15:
        print(f'⚠️ Audio is too long ({ratio:.2f}x). Capping speedup at 1.15x to preserve quality.')
    
    # Apply speed adjustment and pad/trim to exactly match clip_dur
    !ffmpeg -y -i {TTS_RAW} -filter:a "atempo={safe_ratio},apad" -t {clip_dur} -ar 44100 {TTS_ADJ} -loglevel warning
    print(f'✓ Audio adjusted to {clip_dur:.2f}s (Speed applied: {safe_ratio:.2f}x)')
else:
    print('⚠️ Missing files. Skipping speed adjustment.')


**15: Stage 7 — VideoReTalking Lip Sync**

In [ ]:
import sys, os, subprocess
from IPython.display import Video

if 'CLIP' not in globals(): CLIP = 'workspace/clip.mp4'
if 'TTS_ADJ' not in globals(): TTS_ADJ = 'workspace/tts_adjusted.wav'
LIPSYNC = 'workspace/lipsync.mp4'

if os.path.exists(CLIP) and os.path.exists(TTS_ADJ):
    VRT_DIR = os.path.abspath('VideoReTalking')
    vrt_env = os.environ.copy()
    # Essential paths for VideoReTalking sub-modules
    vrt_env['PYTHONPATH'] = ':'.join([
        VRT_DIR,
        os.path.join(VRT_DIR, 'third_part'),
        os.path.join(VRT_DIR, 'third_part', 'face3d'),
        vrt_env.get('PYTHONPATH', '')
    ])

    print('Running VideoReTalking (Stage 7)...')
    result = subprocess.run(
        [sys.executable, 'VideoReTalking/inference.py',
         '--face', CLIP, '--audio', TTS_ADJ, '--outfile', LIPSYNC, '--checkpoint_dir', 'models/VideoReTalking'],
        env=vrt_env, cwd='.'
    )

    if result.returncode != 0:
        print('⚠️ VideoReTalking failed. Falling back to simple mux...')
        # Check if audio is longer than video - if so, freeze last frame
        audio_dur = float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',TTS_ADJ], capture_output=True, text=True).stdout.strip())
        video_dur = float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',CLIP], capture_output=True, text=True).stdout.strip())
        
        if audio_dur > video_dur:
            print(f'Audio is {audio_dur-video_dur:.2f}s longer. Freezing last frame.')
            !ffmpeg -y -i {CLIP} -i {TTS_ADJ} -filter_complex "[0:v]tpad=stop_mode=clone:stop_duration={audio_dur-video_dur}[v]" -map "[v]" -map 1:a -c:v libx264 -c:a aac -shortest {LIPSYNC} -loglevel warning
        else:
            !ffmpeg -y -i {CLIP} -i {TTS_ADJ} -map 0:v -map 1:a -c:v copy -c:a aac -shortest {LIPSYNC} -loglevel warning
        print('✓ Audio mux complete')
    else:
        print('✓ Lip-sync complete!')

if os.path.exists(LIPSYNC):
    display(Video(LIPSYNC, width=640, embed=True))


**16: Stage 8 — GFPGAN Face Enhancement**

In [ ]:
SKIP_ENHANCEMENT = True  # Set to True to skip this slow stage and get output immediately
FINAL = "output/final_dubbed.mp4"
FRAMES_DIR = "workspace/frames_raw"
ENHANCED_DIR = "workspace/gfpgan_out"

import os, subprocess, torch

if SKIP_ENHANCEMENT:
    print("⏩ SKIP_ENHANCEMENT is True. Copying lip-sync video to final output...")
    get_ipython().system(f"cp {LIPSYNC} {FINAL}")
    print(f"✓ Final output (no enhancement): {FINAL}")
else:
    os.makedirs(FRAMES_DIR, exist_ok=True)
    print("Extracting frames...")
    get_ipython().system(f"ffmpeg -y -i {LIPSYNC} -qscale:v 1 -qmin 1 {FRAMES_DIR}/frame_%06d.png -loglevel warning")

    print("Running GFPGAN face restoration...")
    if torch.cuda.is_available():
        PROCESSOR = "cuda"
    elif torch.backends.mps.is_available():
        PROCESSOR = "mps"
    else:
        PROCESSOR = "cpu"
    print(f"Device detected: {PROCESSOR}")
    get_ipython().system(f"python GFPGAN/inference_gfpgan.py --ext png --device {PROCESSOR} -i {FRAMES_DIR} -o {ENHANCED_DIR} -v 1.4 -s 1 --bg_upsampler None")

    fps_r = subprocess.run(["ffprobe","-v","quiet","-select_streams","v:0",
                            "-show_entries","stream=r_frame_rate","-of","csv=p=0", LIPSYNC],
                           capture_output=True, text=True)
    FPS = fps_r.stdout.strip()

    RESTORED = f"{ENHANCED_DIR}/restored_imgs"
    print("Re-encoding final video...")
    get_ipython().system(f"ffmpeg -y -framerate {FPS} -pattern_type glob -i '{RESTORED}/*.png' -i {LIPSYNC} -map 0:v -map 1:a -c:v libx264 -crf 17 -preset slow -c:a aac -shortest {FINAL} -loglevel warning")

    final_dur = get_duration(FINAL)
    print(f"✓ Final output: {FINAL} ({final_dur:.2f}s)")

from IPython.display import Video
Video(FINAL, width=640)


**17: Download Output**

In [ ]:
import os
try:
    from google.colab import files
    files.download(FINAL)
    print(f"✓ Download started for {FINAL}")
except ImportError:
    print(f"✓ Output saved to: {os.path.abspath(FINAL)}")
    print("   (Local machine detected, skipping browser download)")


---
## Pipeline Summary

| Stage | Tool | Duration |
|---|---|---|
| Clip Extract | ffmpeg | ~5s |
| Audio Extract | ffmpeg | ~2s |
| Transcription | Whisper medium | ~30s |
| Translation | IndicTrans2 | ~60s |
| Voice Clone | Coqui XTTS v2 | ~1-3 min |
| Speed Adjust | ffmpeg atempo | ~5s |
| Lip Sync | VideoReTalking | ~10-20s |
| Face Enhance | GFPGAN v1.4 | ~5-7 min |

**Total on free Colab T4: ~8-10 minutes for a 20-second clip**

**Cost: ₹0**